In [ ]:
!pip uninstall -y langchain
!pip install langchain==0.2.16 langchain-community langchain-text-splitters faiss-cpu pypdf groq sentence-transformers

Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
  Using cached langchain-0.2.16-py3-none-any.whl.metadata (7.1 kB)
Using cached langchain-0.2.16-py3-none-any.whl (1.0 MB)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq

In [ ]:
client = Groq(api_key="YOUR_API_KEY")

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Say hello"}]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Expt04.pdf to Expt04.pdf


In [ ]:
file_name = list(uploaded.keys())[0]

loader = PyPDFLoader(file_name)
documents = loader.load()

In [ ]:
print(len(documents))
print(documents[0].page_content[:300])

1
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives: 
1. To understand RAG architecture 
2. To build a retrieval-based QA chatbot 
 
 
Key Concepts: Do


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

In [ ]:
print(len(docs))
print(docs[0].page_content[:300])

2
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives: 
1. To understand RAG architecture 
2. To build a retrieval-based QA chatbot 
 
 
Key Concepts: Do


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_3954/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
results = retriever.get_relevant_documents("main topic of document")

for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:200])


Result 1:
5. LLM: Generates final response using retrieved context. 
 
Algorithm 
1. Install required Python libraries. 
2. Upload and load document. 
3. Load the PDF Document. 
4. Apply Text Chunking. 
5. Crea

Result 2:
T.Y.B.Tech CSE(AIML) Subject: Agentic Systems Lab 
  
Experiment No. : 4 
 
Title: Design and development of a Retrieval-Augmented Generation (RAG) based question 
answering chat system. 
 
Objectives


/tmp/ipykernel_3954/175830645.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  results = retriever.get_relevant_documents("main topic of document")


In [ ]:
def research_assistant(query):

    # Step 1: Retrieve relevant chunks
    docs = retriever.get_relevant_documents(query)

    context = "\n\n".join([doc.page_content for doc in docs])

    # Step 2: Send to LLM
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are an AI research assistant. Answer ONLY using the given context."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
print(research_assistant("What is this document about?"))

This document is about the design and development of a Retrieval-Augmented Generation (RAG) based question answering chat system. The experiment aims to understand the RAG architecture and build a retrieval-based QA chatbot.


In [ ]:
while True:
    query = input("You: ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    print("Assistant:", research_assistant(query))
    print()

You: What is JS
Assistant: Based on the provided context, it seems like 'JS' is not explicitly mentioned as a component or concept in Retrieval-Augmented Generation (RAG) architecture. However, given the AIML (Artificial Intelligence and Machine Learning) context, 'JS' might refer to JavaScript, a high-level, dynamic, and interpreted programming language that is commonly used for client-side scripting on the web.

In the context of this experiment (Experiment No. 4), 'JS' is unlikely to be a relevant concept. However, if you would like to proceed with JavaScript in your project, you might need to use a JavaScript library like 'py-js' or 'js2py' to integrate JavaScript code with your Python implementation. 

However, based on the given algorithm and experiment requirements, it is most likely that 'JS' was a typo or a mistake, and the context actually refers to the 'JavaScript' was not relevant here as it is more of the client side  implementation whereas this experiment focuses more on 